In [1]:
# Install modules
%pip install nltk

# Import necessary libraries
import pandas as pd
import random
import re
import nltk
import torch

nltk.download('wordnet')

from transformers import BertTokenizer
from nltk.corpus import wordnet
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

# Load dataset (ONLY train.csv now)
train_data = pd.read_csv('data/tweet_emotion_intensity/train.csv')

# Function to clean the text
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    return text

# Clean text
train_data['cleaned_text'] = train_data['tweet'].apply(clean_text)
train_data['cleaned_text'].fillna('unknown', inplace=True)

# Map labels
def map_sentiment(value):
    value = str(value).strip().lower()

    if value == "high":
        return 1
    elif value == "medium":
        return 0.5
    elif value == "low":
        return 0
    else:
        return None

train_data['sentiment_intensity'] = train_data['sentiment_intensity'].apply(map_sentiment)

# Drop invalid rows
train_data = train_data.dropna(subset=['sentiment_intensity']).reset_index(drop=True)

# ------------------------
# SPLIT INTO TRAIN / VAL / TEST
# ------------------------
train_df, temp_df = train_test_split(
    train_data,
    test_size=0.3,
    random_state=42,
    stratify=train_data['sentiment_intensity']
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df['sentiment_intensity']
)

# ------------------------
# AUGMENTATION (TRAIN ONLY)
# ------------------------
def synonym_replacement(word):
    synonyms = wordnet.synsets(word)
    if synonyms:
        return random.choice(synonyms).lemmas()[0].name()
    return word

def augment_text(text):
    words = text.split()
    augmented_words = [
        synonym_replacement(word) if random.random() > 0.8 else word
        for word in words
    ]
    return ' '.join(augmented_words)

train_df['augmented_text'] = train_df['cleaned_text'].apply(augment_text)

# ------------------------
# TOKENIZATION
# ------------------------
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

train_tokens = tokenizer(
    train_df['augmented_text'].tolist(),
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors='pt'
)

val_tokens = tokenizer(
    val_df['cleaned_text'].tolist(),
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors='pt'
)

test_tokens = tokenizer(
    test_df['cleaned_text'].tolist(),
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors='pt'
)

# ------------------------
# TENSORS
# ------------------------
train_input_ids = train_tokens['input_ids']
val_input_ids = val_tokens['input_ids']
test_input_ids = test_tokens['input_ids']

train_attention_masks = train_tokens['attention_mask']
val_attention_masks = val_tokens['attention_mask']
test_attention_masks = test_tokens['attention_mask']

train_labels = torch.tensor(train_df['sentiment_intensity'].tolist())
val_labels = torch.tensor(val_df['sentiment_intensity'].tolist())
test_labels = torch.tensor(test_df['sentiment_intensity'].tolist())

# ------------------------
# DATASETS
# ------------------------
train_dataset = TensorDataset(train_input_ids, train_attention_masks, train_labels)
val_dataset = TensorDataset(val_input_ids, val_attention_masks, val_labels)
test_dataset = TensorDataset(test_input_ids, test_attention_masks, test_labels)

# ------------------------
# DATALOADERS
# ------------------------
train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=16)
test_dataloader = DataLoader(test_dataset, batch_size=16)

print("Train / Validation / Test sets created successfully!")


[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: /anaconda/envs/azureml_py310_sdkv2/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Train / Validation / Test sets created successfully!


[nltk_data] Downloading package wordnet to
[nltk_data]     /home/azureuser/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
